# Analisis AS-level de Chile (BGP, RIPE Atlas y combinado)

Este notebook responde:

1. Como esta conformada la topologia AS-level de Chile.
2. Que ASes e interconexiones son mas relevantes.
3. Diferencias entre BGP y traceroute/RIPE Atlas.
4. Que tan completa es la topologia al combinar fuentes.
5. Que ASNs y enlaces son consistentes en ambas fuentes.
6. Que aporta usar una topologia combinada.
7. Que ASNs chilenos pueden considerarse estrategicos.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import plotly.express as px

DATA_DIR = Path('../data/csv') if Path('../data/csv').exists() else Path('data/csv')
print('DATA_DIR:', DATA_DIR.resolve())

DATA_DIR: /home/vale/Escritorio/GIT/as-topology-visualizer-cl/data/csv


In [2]:
def load_dataset(name):
    nodes = pd.read_csv(DATA_DIR / name / 'nodes.csv')
    edges = pd.read_csv(DATA_DIR / name / 'edges.csv')
    nodes['name'] = nodes['name'].fillna('')
    nodes['degree'] = nodes['in_degree'] + nodes['out_degree']
    return nodes, edges


def gini(values):
    arr = np.array(values, dtype=float)
    arr = arr[arr >= 0]
    if arr.size == 0 or arr.sum() == 0:
        return 0.0
    arr.sort()
    n = arr.size
    return float((2 * np.sum((np.arange(n) + 1) * arr)) / (n * arr.sum()) - (n + 1) / n)


def asn_edge_set(nodes, edges):
    id2asn = dict(zip(nodes['node_id'], nodes['asn']))
    return {(int(id2asn[s]), int(id2asn[d])) for s, d in edges[['src_id', 'dst_id']].itertuples(index=False)}


def jaccard(a, b):
    u = a | b
    return len(a & b) / len(u) if u else 0.0


def infer_chile_asns(nodes):
    explicit = set(nodes.loc[nodes['name'].str.contains(r'\bchile\b', case=False, regex=True), 'asn'].astype(int))
    manual = {27986, 6568, 27925, 27651, 22047, 52341, 18822, 14117, 10834, 20015, 6471, 6429, 14259, 27678, 20191, 23140, 64112}
    return explicit | manual

In [3]:
datasets = {
    'BGP': load_dataset('bgp'),
    'RIPE Atlas': load_dataset('ripe_atlas'),
    'Merged': load_dataset('merged'),
}

rows = []
for name, (n, e) in datasets.items():
    max_edges = len(n) * (len(n) - 1)
    rows.append({
        'dataset': name,
        'nodes': len(n),
        'edges': len(e),
        'density': len(e) / max_edges if max_edges else 0.0,
        'avg_degree': n['degree'].mean(),
        'median_degree': n['degree'].median(),
        'p90_degree': n['degree'].quantile(0.9),
        'max_degree': n['degree'].max(),
        'gini_degree': gini(n['degree']),
    })
summary = pd.DataFrame(rows).set_index('dataset')
display(summary.round(4))

KeyError: 'name'

## 1) Como esta conformada la topologia AS-level de Chile

In [4]:
merged_nodes, merged_edges = datasets['Merged']

stub_share = 100 * (merged_nodes['degree'] <= 2).mean()
core_thr = merged_nodes['degree'].quantile(0.9)

a1 = [
    f"- Merged contiene {len(merged_nodes):,} ASNs y {len(merged_edges):,} enlaces.",
    f"- Gini de grado: {gini(merged_nodes['degree']):.3f} (alta concentracion).",
    f"- ASNs de borde (grado <=2): {stub_share:.1f}%.",
    f"- Umbral de nucleo (p90): grado >= {core_thr:.0f}.",
]
display(Markdown("\n".join(a1)))

- Merged contiene 9,839 ASNs y 29,649 enlaces.
- Gini de grado: 0.689 (alta concentracion).
- ASNs de borde (grado <=2): 57.0%.
- Umbral de nucleo (p90): grado >= 7.

## 2) ASes e interconexiones mas relevantes

In [5]:
# Ranking de ASNs criticos (score degree + path_occurrences)
crit = merged_nodes.copy()
crit['score'] = 0.5 * (crit['degree'] / max(crit['degree'].max(), 1)) + 0.5 * (crit['path_occurrences'] / max(crit['path_occurrences'].max(), 1))
crit = crit.sort_values(['score', 'degree', 'path_occurrences'], ascending=False)
display(crit[['asn', 'name', 'degree', 'in_degree', 'out_degree', 'path_occurrences', 'score']].head(20))

id2asn = dict(zip(merged_nodes['node_id'], merged_nodes['asn']))
asn2name = dict(zip(merged_nodes['asn'], merged_nodes['name']))
top_asns = set(crit.head(20)['asn'])

ed = merged_edges.copy()
ed['src_asn'] = ed['src_id'].map(id2asn)
ed['dst_asn'] = ed['dst_id'].map(id2asn)
key_edges = ed[(ed['src_asn'].isin(top_asns)) | (ed['dst_asn'].isin(top_asns))]
key_edges = key_edges.groupby(['src_asn', 'dst_asn'], as_index=False)['weight'].sum().sort_values('weight', ascending=False).head(20)
key_edges['src_name'] = key_edges['src_asn'].map(asn2name)
key_edges['dst_name'] = key_edges['dst_asn'].map(asn2name)
display(key_edges[['src_asn', 'src_name', 'dst_asn', 'dst_name', 'weight']])

,asn,name,degree,in_degree,out_degree,path_occurrences,score
430,6939,Hurricane Electric,5676,31,5645,11220414,0.862316
2997,34549,meerfarbig GmbH & Co. KG,339,13,326,15484305,0.529863
90,1299,Arelion (Twelve99),845,18,827,11752360,0.453929
3024,34927,FogNet - iFog GmbH,538,19,519,8505727,0.322049
4861,60150,area-7 IT-Services GmbH,357,6,351,8809975,0.315929
15,174,"Cogent Communications, Inc.",1405,22,1383,5655654,0.306392
6757,213151,AS213151 LLC,300,6,294,5126755,0.191974
599,8888,xTom Pty Ltd,354,7,347,4959593,0.191333
4062,51019,Ljósnet VPC,72,7,65,5175467,0.173462
169,2914,NTT Global IP Network,488,16,472,3697598,0.162386


,src_asn,src_name,dst_asn,dst_name,weight
15049,60150,area-7 IT-Services GmbH,34549,meerfarbig GmbH & Co. KG,5293043
14014,51019,Ljósnet VPC,34927,FogNet - iFog GmbH,4422870
11596,8888,xTom Pty Ltd,1299,Arelion (Twelve99),3205994
15427,213151,AS213151 LLC,34549,meerfarbig GmbH & Co. KG,2608961
12939,34549,meerfarbig GmbH & Co. KG,174,"Cogent Communications, Inc.",2048239
13272,34927,FogNet - iFog GmbH,1299,Arelion (Twelve99),1882288
12952,34549,meerfarbig GmbH & Co. KG,2914,NTT Global IP Network,1745083
12942,34549,meerfarbig GmbH & Co. KG,1299,Arelion (Twelve99),1137694
15104,60150,area-7 IT-Services GmbH,48314,IP-Projects GmbH & Co. KG,1024090
15454,213151,AS213151 LLC,41051,FREETRANSIT,906412


## 3) Diferencias BGP vs RIPE Atlas

In [6]:
bgp_nodes, bgp_edges = datasets['BGP']
ripe_nodes, ripe_edges = datasets['RIPE Atlas']

asn_b = set(bgp_nodes['asn'].astype(int))
asn_r = set(ripe_nodes['asn'].astype(int))
edge_b = asn_edge_set(bgp_nodes, bgp_edges)
edge_r = asn_edge_set(ripe_nodes, ripe_edges)

common_asn = asn_b & asn_r
common_edge = edge_b & edge_r

# Correlacion de grado en ASNs comunes + organizacion de referencia
bgp_deg = dict(zip(bgp_nodes['asn'], bgp_nodes['degree']))
ripe_deg = dict(zip(ripe_nodes['asn'], ripe_nodes['degree']))

bgp_org = dict(zip(bgp_nodes['asn'], bgp_nodes['name'].fillna('').astype(str)))
ripe_org = dict(zip(ripe_nodes['asn'], ripe_nodes['name'].fillna('').astype(str)))

bgp_org_id = dict(zip(bgp_nodes['asn'], bgp_nodes.get('org_id', pd.Series([np.nan] * len(bgp_nodes)))))
ripe_org_id = dict(zip(ripe_nodes['asn'], ripe_nodes.get('org_id', pd.Series([np.nan] * len(ripe_nodes)))))

cd = pd.DataFrame({'asn': sorted(common_asn)})
cd['deg_bgp'] = cd['asn'].map(bgp_deg)
cd['deg_ripe'] = cd['asn'].map(ripe_deg)
cd['org_bgp'] = cd['asn'].map(bgp_org).fillna('')
cd['org_ripe'] = cd['asn'].map(ripe_org).fillna('')
cd['org_id_bgp'] = cd['asn'].map(bgp_org_id)
cd['org_id_ripe'] = cd['asn'].map(ripe_org_id)

# Nombre de organizacion preferido (RIPE -> BGP -> fallback)
cd['organizacion'] = (
    cd['org_ripe'].replace('', np.nan)
      .fillna(cd['org_bgp'].replace('', np.nan))
      .fillna('(sin nombre)')
)

pearson = cd['deg_bgp'].corr(cd['deg_ripe']) if len(cd) > 1 else np.nan

lines = [
    f"- ASNs BGP: {len(asn_b):,}",
    f"- ASNs RIPE Atlas: {len(asn_r):,}",
    f"- ASNs comunes: {len(common_asn):,} (Jaccard={jaccard(asn_b, asn_r):.4f})",
    f"- Enlaces comunes: {len(common_edge):,} (Jaccard={jaccard(edge_b, edge_r):.6f})",
    f"- Correlacion de grado en comunes (Pearson): {pearson:.3f}",
]
display(Markdown("\n".join(lines)))

display(
    cd.sort_values(['deg_ripe', 'deg_bgp'], ascending=False)
      .head(20)[['asn', 'organizacion', 'org_bgp', 'org_ripe', 'org_id_bgp', 'org_id_ripe', 'deg_bgp', 'deg_ripe']]
)


- ASNs BGP: 16,988
- ASNs RIPE Atlas: 127
- ASNs comunes: 122 (Jaccard=0.0072)
- Enlaces comunes: 90 (Jaccard=0.000188)
- Correlacion de grado en comunes (Pearson): 0.221

,asn,organizacion,org_bgp,org_ripe,org_id_bgp,org_id_ripe,deg_bgp,deg_ripe
31,14259,GTD Chile,GTD Chile,GTD Chile,1046.0,1046.0,156,42
12,6429,CLARO CHILE AS6429,CLARO CHILE AS6429,CLARO CHILE AS6429,25911.0,25911.0,31,34
29,13335,Cloudflare,Cloudflare,Cloudflare,4715.0,4715.0,1631,30
16,6939,Hurricane Electric,Hurricane Electric,Hurricane Electric,396.0,396.0,10225,29
17,7004,CTC Transmisiones Regionales S.A.,CTC Transmisiones Regionales S.A.,CTC Transmisiones Regionales S.A.,15675.0,15675.0,60,25
69,52304,(sin nombre),,,NaN,NaN,6,25
67,52234,(sin nombre),,,NaN,NaN,11,24
9,3549,Lumen AS 3549,Lumen AS 3549,Lumen AS 3549,682.0,682.0,145,20
8,3356,Lumen AS3356,Lumen AS3356,Lumen AS3356,682.0,682.0,2525,19
33,15208,(sin nombre),,,NaN,NaN,4,18


## 4) Completitud al combinar fuentes

In [7]:
asn_union = asn_b | asn_r
edge_union = edge_b | edge_r

merged_asn = set(merged_nodes['asn'].astype(int))
merged_edge = asn_edge_set(merged_nodes, merged_edges)

lines = [
    f"- Union teorica BGP U RIPE: {len(asn_union):,} ASNs y {len(edge_union):,} enlaces.",
    f"- Snapshot Merged (CSV actual): {len(merged_asn):,} ASNs y {len(merged_edge):,} enlaces.",
    "- Si hay diferencia, suele deberse a filtros/metodologia de preprocesamiento.",
]
display(Markdown("\n".join(lines)))

- Union teorica BGP U RIPE: 16,993 ASNs y 478,343 enlaces.
- Snapshot Merged (CSV actual): 9,839 ASNs y 29,649 enlaces.
- Si hay diferencia, suele deberse a filtros/metodologia de preprocesamiento.

## 5) ASNs y enlaces consistentes en ambas fuentes

In [8]:
common = pd.DataFrame({'asn': sorted(common_asn)})
common = common.merge(
    bgp_nodes[['asn', 'name', 'degree', 'path_occurrences']].rename(columns={'name': 'name_bgp', 'degree': 'deg_bgp', 'path_occurrences': 'path_bgp'}),
    on='asn', how='left'
).merge(
    ripe_nodes[['asn', 'name', 'degree', 'path_occurrences']].rename(columns={'name': 'name_ripe', 'degree': 'deg_ripe', 'path_occurrences': 'path_ripe'}),
    on='asn', how='left'
)
common['name'] = common['name_ripe'].replace('', np.nan).fillna(common['name_bgp']).fillna('')

for c in ['deg_bgp', 'deg_ripe', 'path_bgp', 'path_ripe']:
    common[c + '_n'] = common[c] / max(common[c].max(), 1)

common['consistency_score'] = 0.5 * (common['deg_bgp_n'] + common['deg_ripe_n']) / 2 + 0.5 * (common['path_bgp_n'] + common['path_ripe_n']) / 2
display(common.sort_values('consistency_score', ascending=False)[['asn', 'name', 'deg_bgp', 'deg_ripe', 'path_bgp', 'path_ripe', 'consistency_score']].head(20))

# Enlaces comunes con peso BGP y RIPE
id2asn_b = dict(zip(bgp_nodes['node_id'], bgp_nodes['asn']))
id2asn_r = dict(zip(ripe_nodes['node_id'], ripe_nodes['asn']))
eb = {(int(id2asn_b[s]), int(id2asn_b[d])): int(w) for s, d, w in bgp_edges[['src_id', 'dst_id', 'weight']].itertuples(index=False)}
er = {(int(id2asn_r[s]), int(id2asn_r[d])): int(w) for s, d, w in ripe_edges[['src_id', 'dst_id', 'weight']].itertuples(index=False)}

nm = pd.concat([bgp_nodes[['asn', 'name']], ripe_nodes[['asn', 'name']]], ignore_index=True).drop_duplicates('asn')
nm = dict(zip(nm['asn'], nm['name'].fillna('')))
rows = []
for a, b in sorted(common_edge):
    rows.append({
        'src_asn': a,
        'src_name': nm.get(a, ''),
        'dst_asn': b,
        'dst_name': nm.get(b, ''),
        'weight_bgp': eb[(a, b)],
        'weight_ripe': er[(a, b)],
        'shared_strength': min(eb[(a, b)], er[(a, b)]),
    })
common_edges_df = pd.DataFrame(rows).sort_values(['shared_strength', 'weight_ripe'], ascending=False)
display(common_edges_df.head(20))

,asn,name,deg_bgp,deg_ripe,path_bgp,path_ripe,consistency_score
16,6939,Hurricane Electric,10225,29,1181357706,56,0.695677
31,14259,GTD Chile,156,42,10186057,312,0.505782
8,3356,Lumen AS3356,2525,19,1254434512,66,0.470017
0,174,"Cogent Communications, Inc.",3489,8,1053036986,9,0.343537
4,1299,Arelion (Twelve99),2059,4,1294291363,6,0.328960
29,13335,Cloudflare,1631,30,61832025,67,0.284078
12,6429,CLARO CHILE AS6429,31,34,10870805,95,0.281360
17,7004,CTC Transmisiones Regionales S.A.,60,25,7535477,132,0.257501
66,49544,i3D.net,8349,2,172586966,2,0.250976
59,36236,NetActuate,7855,7,21628482,8,0.244308


,src_asn,src_name,dst_asn,dst_name,weight_bgp,weight_ripe,shared_strength
53,14259,GTD Chile,15208,,3213,78,78
16,3356,Lumen AS3356,14259,GTD Chile,673829,28,28
30,6939,Hurricane Electric,263237,PowerHost Chile,1506205,28,28
34,7004,CTC Transmisiones Regionales S.A.,14259,GTD Chile,3418,24,24
46,12956,Telxius Cable,13335,Cloudflare,166425,23,23
41,7418,Movistar ISP,12956,Telxius Cable,42,15,15
28,6762,Telecom Italia Sparkle,14259,GTD Chile,2063252,14,14
60,18747,IFX,15208,,113,13,13
50,12956,Telxius Cable,36351,"SoftLayer Technologies, Inc. (an IBM Company)",11231,9,9
19,3549,Lumen AS 3549,3356,Lumen AS3356,14892011,8,8


## 6) Aporte de una topologia combinada y 7) ASNs chilenos estrategicos

In [9]:
only_b = asn_b - asn_r
only_r = asn_r - asn_b
only_eb = edge_b - edge_r
only_er = edge_r - edge_b

lines = [
    f"- Solo BGP: {len(only_b):,} ASNs y {len(only_eb):,} enlaces.",
    f"- Solo RIPE: {len(only_r):,} ASNs y {len(only_er):,} enlaces.",
    "- Conclusiones: BGP aporta cobertura; RIPE aporta validacion operacional.",
]
display(Markdown("\n".join(lines)))

chile_asn = infer_chile_asns(merged_nodes)
cl = merged_nodes[merged_nodes['asn'].isin(chile_asn)].copy()
cl['strategic_score'] = 0.5 * (cl['degree'] / max(cl['degree'].max(), 1)) + 0.5 * (cl['path_occurrences'] / max(cl['path_occurrences'].max(), 1))
display(cl.sort_values(['strategic_score', 'degree', 'path_occurrences'], ascending=False)[['asn', 'name', 'degree', 'path_occurrences', 'strategic_score']].head(20))

- Solo BGP: 16,866 ASNs y 478,014 enlaces.
- Solo RIPE: 5 ASNs y 239 enlaces.
- Conclusiones: BGP aporta cobertura; RIPE aporta validacion operacional.

,asn,name,degree,path_occurrences,strategic_score
1129,14259,GTD Chile,62,54976,0.944437
7155,263237,PowerHost Chile,31,43825,0.604290
2426,27986,entel IP Internacional,7,61849,0.556452
386,6429,CLARO CHILE AS6429,35,7093,0.339599
390,6471,entel Chile Servicios Fijos,24,6842,0.248861
2347,27651,entel MPLS,3,25695,0.231917
1814,22047,VTR Global COM S.A.,4,18587,0.182519
1109,14117,Telefonica del Sur,8,13952,0.177307
1544,18822,Gtd Manquehue,2,10442,0.100544
5210,64112,PIT Chile - Transit,12,353,0.099628


In [10]:
tmp = summary.reset_index()
fig1 = px.bar(tmp, x='dataset', y='nodes', title='ASNs por fuente')
fig2 = px.bar(tmp, x='dataset', y='edges', title='Enlaces por fuente')
fig1.show()
fig2.show()

## Como se esta haciendo el merge (BGP + RIPE Atlas)

La vista `Combinado` que usamos en este repo se interpreta como una fusion a nivel AS con estos pasos conceptuales:

1. **Normalizacion por ASN**
   Se usan los ASNs como clave comun para alinear nodos de BGP y RIPE Atlas.

2. **Union de nodos y aristas**
   Se construye el conjunto combinado con la logica de union:
   - `ASNs_combinado = ASNs_BGP ∪ ASNs_RIPE`
   - `enlaces_combinado = enlaces_BGP ∪ enlaces_RIPE`

3. **Remapeo de identificadores internos (`node_id`)**
   Al consolidar por ASN, los `node_id` de cada fuente se remapean a un espacio comun para producir un solo grafo coherente.

4. **Consolidacion de atributos**
   Para cada ASN se conservan/combinen atributos disponibles (`name`, `path_occurrences`, `ix_count`, etc.).
   Cuando hay conflicto o faltantes entre fuentes, se prioriza el valor disponible segun el pipeline de preprocesamiento.

5. **Resultado final**
   El CSV `data/csv/merged/*` representa esa fusion ya materializada y es la base de analisis en este notebook.

> Nota: en este repositorio se dispone del resultado final del merge (CSV), no del script historico completo de construccion paso a paso.
